In [39]:
# Husayn El Sharif
# Merge data from hourly pudl and open-meteo API for SOCO region
# Perform feature engineering
# Output "data/soco_region_hourly_demand_weather_features_dataset.csv"

In [40]:
# imports
import os
import pandas as pd
import numpy as np
import holidays

import re
import pandas as pd
from pathlib import Path


In [41]:
# data paths
pudl_data_path = "data/soco_hourly_operations.csv"
weather_data_path = "data/soco_region_hourly_weather.csv"

# output path: hourly demand, weather, and engineered features (along with target variable)
merged_data_path = "data/soco_modeling_dataset.csv"

In [42]:
# import data
pudl_df = pd.read_csv(pudl_data_path)
weather_df = pd.read_csv(weather_data_path)

In [43]:
# for Public Utility Data Liberation (PUDL) data, only keep the columns: 
# datetime_utc: the timestamp of the data point
# demand_imputed_pudl_mwh: the imputed (cleaned/filled) energy demand in megawatt-hours
pudl_df_subset = pudl_df[['datetime_utc', 'demand_imputed_pudl_mwh']]

In [44]:
# join data on datetime_utc with pudl_df_subset as left and weather_df as right
merged_df = pd.merge(pudl_df_subset, weather_df, on="datetime_utc", how="left")

In [45]:
merged_df

,datetime_utc,demand_imputed_pudl_mwh,temperature_2m_albany_ga,temperature_2m_atlanta_ga,temperature_2m_birmingham_al,temperature_2m_huntsville_al,temperature_2m_meridian_ms,temperature_2m_mobile_al,temperature_2m_savannah_ga,relative_humidity_2m_albany_ga,...,wind_speed_10m_meridian_ms,wind_speed_10m_mobile_al,wind_speed_10m_savannah_ga,shortwave_radiation_albany_ga,shortwave_radiation_atlanta_ga,shortwave_radiation_birmingham_al,shortwave_radiation_huntsville_al,shortwave_radiation_meridian_ms,shortwave_radiation_mobile_al,shortwave_radiation_savannah_ga
0,2015-07-01 06:00:00+00:00,24823.000,23.4,20.3,21.0,21.0,23.4,25.7,24.2,96,...,11.7,12.9,16.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-07-01 07:00:00+00:00,24121.000,23.2,20.9,21.1,22.6,23.5,24.9,23.5,85,...,12.5,11.9,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-07-01 08:00:00+00:00,23383.000,22.8,20.6,20.9,22.2,23.4,24.6,23.4,88,...,11.7,9.8,19.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-07-01 09:00:00+00:00,23184.000,22.6,20.1,20.8,21.5,23.1,24.2,23.3,89,...,10.7,6.9,19.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-07-01 10:00:00+00:00,23680.000,22.5,19.9,20.8,21.1,22.8,23.7,23.2,91,...,10.2,6.1,16.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92828,2026-02-01 02:00:00+00:00,26530.680,-4.2,-6.6,-5.2,-6.0,-4.8,-2.5,-2.8,41,...,17.9,21.4,29.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92829,2026-02-01 03:00:00+00:00,26132.540,-4.8,-6.8,-5.9,-6.2,-5.1,-2.9,-2.7,43,...,18.7,21.5,26.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92830,2026-02-01 04:00:00+00:00,25221.031,-5.1,-6.9,-6.4,-6.6,-5.4,-3.4,-3.5,45,...,18.8,20.8,22.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92831,2026-02-01 05:00:00+00:00,24196.906,-5.1,-6.9,-6.7,-6.9,-5.8,-3.9,-3.2,45,...,16.0,20.5,25.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
# add a column with local time by converting datetime_utc to datetime and then to local time (EST)
merged_df['datetime_local'] = pd.to_datetime(merged_df['datetime_utc'], utc=True).dt.tz_convert('US/Eastern')

# make datetime_local the second column and datetime_utc the first column
cols = merged_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('datetime_local')))
merged_df = merged_df[cols]

# make sure 'demand_imputed_pudl_mwh' is float and round to 0 decimal places
merged_df["demand_imputed_pudl_mwh"] = (
    merged_df["demand_imputed_pudl_mwh"]
    .astype(float)
    .round(0)
)

# make deep copy of merged_df to avoid SettingWithCopyWarning
merged_df = merged_df.copy()

In [47]:
# enrich merged_df with additional time features

dt = merged_df["datetime_local"]

# ======================
# Calendar features
# ======================
merged_df["hour"] = dt.dt.hour
merged_df["day_of_week"] = dt.dt.dayofweek
merged_df["day_of_month"] = dt.dt.day
merged_df["day_of_year"] = dt.dt.dayofyear
merged_df["month"] = dt.dt.month
merged_df["quarter"] = dt.dt.quarter
merged_df["year"] = dt.dt.year
merged_df["week_of_year"] = dt.dt.isocalendar().week.astype(int)

# ======================
# Flags
# ======================
merged_df["is_weekend"] = (merged_df["day_of_week"] >= 5).astype(int)
merged_df["is_month_start"] = dt.dt.is_month_start.astype(int)
merged_df["is_month_end"] = dt.dt.is_month_end.astype(int)
merged_df["is_quarter_start"] = dt.dt.is_quarter_start.astype(int)
merged_df["is_quarter_end"] = dt.dt.is_quarter_end.astype(int)

# ======================
# Cyclical encodings
# ======================
merged_df["hour_sin"] = np.sin(2 * np.pi * merged_df["hour"] / 24)
merged_df["hour_cos"] = np.cos(2 * np.pi * merged_df["hour"] / 24)

merged_df["day_of_week_sin"] = np.sin(2 * np.pi * merged_df["day_of_week"] / 7)
merged_df["day_of_week_cos"] = np.cos(2 * np.pi * merged_df["day_of_week"] / 7)

merged_df["month_sin"] = np.sin(2 * np.pi * merged_df["month"] / 12)
merged_df["month_cos"] = np.cos(2 * np.pi * merged_df["month"] / 12)

merged_df["day_of_year_sin"] = np.sin(2 * np.pi * merged_df["day_of_year"] / 365.25)
merged_df["day_of_year_cos"] = np.cos(2 * np.pi * merged_df["day_of_year"] / 365.25)

# ======================
# US Observed Holidays (optimized)
# ======================

# Precompute observed US holidays over your date range
years = merged_df["datetime_local"].dt.year.unique()
us_holidays_observed = holidays.US(years=years, observed=True)

# Vectorized check
merged_df["is_holiday"] = (
    merged_df["datetime_local"].dt.date.isin(us_holidays_observed)
).astype(int)

In [48]:
# Make sure datetime_utc is datetime type and sorted
merged_df["datetime_utc"] = pd.to_datetime(merged_df["datetime_utc"])
merged_df = merged_df.sort_values("datetime_utc").reset_index(drop=True)

# Check timestamp gaps
gaps = merged_df["datetime_utc"].diff().dt.total_seconds()

is_hourly_continuous = (gaps.dropna() == 3600).all()
is_sorted = merged_df["datetime_utc"].is_monotonic_increasing
has_no_duplicates = merged_df["datetime_utc"].duplicated().sum() == 0

print("Hourly continuous:", is_hourly_continuous)
print("Sorted increasing:", is_sorted)
print("No duplicate timestamps:", has_no_duplicates)

Hourly continuous: True
Sorted increasing: True
No duplicate timestamps: True


In [49]:
# create lagged values of demand_imputed_pudl_mwh for hours: 1,2,3, 24, 48, 72, 168
for hour in [1, 2, 3, 24, 48, 72, 168]:
    merged_df[f'demand_imputed_pudl_mwh_lag_{hour}h'] = merged_df['demand_imputed_pudl_mwh'].shift(hour)

# create rolling mean demand features for hours 3, 6, 24, 168
for hour in [3, 6, 24, 168]:
    merged_df[f'demand_imputed_pudl_mwh_rolling_mean_{hour}h'] = merged_df['demand_imputed_pudl_mwh'].shift(1).rolling(window=hour).mean()

# create rolling max, min, std demand features for previous 24 hours
for hour in [24]:
    merged_df[f'demand_imputed_pudl_mwh_rolling_max_{hour}h'] = merged_df['demand_imputed_pudl_mwh'].shift(1).rolling(window=hour).max()
    merged_df[f'demand_imputed_pudl_mwh_rolling_min_{hour}h'] = merged_df['demand_imputed_pudl_mwh'].shift(1).rolling(window=hour).min()
    merged_df[f'demand_imputed_pudl_mwh_rolling_std_{hour}h'] = merged_df['demand_imputed_pudl_mwh'].shift(1).rolling(window=hour).std()

In [50]:
merged_df

,datetime_utc,datetime_local,demand_imputed_pudl_mwh,temperature_2m_albany_ga,temperature_2m_atlanta_ga,temperature_2m_birmingham_al,temperature_2m_huntsville_al,temperature_2m_meridian_ms,temperature_2m_mobile_al,temperature_2m_savannah_ga,...,demand_imputed_pudl_mwh_lag_48h,demand_imputed_pudl_mwh_lag_72h,demand_imputed_pudl_mwh_lag_168h,demand_imputed_pudl_mwh_rolling_mean_3h,demand_imputed_pudl_mwh_rolling_mean_6h,demand_imputed_pudl_mwh_rolling_mean_24h,demand_imputed_pudl_mwh_rolling_mean_168h,demand_imputed_pudl_mwh_rolling_max_24h,demand_imputed_pudl_mwh_rolling_min_24h,demand_imputed_pudl_mwh_rolling_std_24h
0,2015-07-01 06:00:00+00:00,2015-07-01 02:00:00-04:00,24823.0,23.4,20.3,21.0,21.0,23.4,25.7,24.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-07-01 07:00:00+00:00,2015-07-01 03:00:00-04:00,24121.0,23.2,20.9,21.1,22.6,23.5,24.9,23.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-07-01 08:00:00+00:00,2015-07-01 04:00:00-04:00,23383.0,22.8,20.6,20.9,22.2,23.4,24.6,23.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-07-01 09:00:00+00:00,2015-07-01 05:00:00-04:00,23184.0,22.6,20.1,20.8,21.5,23.1,24.2,23.3,...,NaN,NaN,NaN,24109.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-07-01 10:00:00+00:00,2015-07-01 06:00:00-04:00,23680.0,22.5,19.9,20.8,21.1,22.8,23.7,23.2,...,NaN,NaN,NaN,23562.666667,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92828,2026-02-01 02:00:00+00:00,2026-01-31 21:00:00-05:00,26531.0,-4.2,-6.6,-5.2,-6.0,-4.8,-2.5,-2.8,...,33471.0,34417.0,29256.0,26038.333333,24497.500000,27799.000000,32407.130952,31240.0,22573.0,2797.599685
92829,2026-02-01 03:00:00+00:00,2026-01-31 22:00:00-05:00,26133.0,-4.8,-6.8,-5.9,-6.2,-5.1,-2.9,-2.7,...,33526.0,34295.0,28628.0,26418.000000,25062.166667,27609.500000,32390.910714,31240.0,22573.0,2718.683745
92830,2026-02-01 04:00:00+00:00,2026-01-31 23:00:00-05:00,25221.0,-5.1,-6.9,-6.4,-6.6,-5.4,-3.4,-3.5,...,32905.0,33644.0,27812.0,26405.666667,25655.500000,27396.708333,32376.059524,30842.0,22573.0,2620.250112
92831,2026-02-01 05:00:00+00:00,2026-02-01 00:00:00-05:00,24197.0,-5.1,-6.9,-6.7,-6.9,-5.8,-3.9,-3.2,...,32271.0,32940.0,27056.0,25961.666667,26000.000000,27162.500000,32360.636905,30427.0,22573.0,2549.156143


In [51]:
# Create regional average weather features
weather_vars = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "surface_pressure",
    "wind_speed_10m",
    "shortwave_radiation",
]

for var in weather_vars:
    cols = [col for col in merged_df.columns if col.startswith(var + "_")]
    merged_df[f"{var}_regional_mean"] = merged_df[cols].mean(axis=1)

In [52]:
# drop rows with NaN due to lagging and rolling operations
target_col = "demand_imputed_pudl_mwh"

lag_roll_cols = [
    col for col in merged_df.columns
    if "lag" in col or "rolling" in col
]

merged_df = merged_df.dropna(subset=lag_roll_cols + [target_col]).copy()

In [53]:
# list of columns in merged_df
for col in merged_df.columns:
    print(col)

datetime_utc
datetime_local
demand_imputed_pudl_mwh
temperature_2m_albany_ga
temperature_2m_atlanta_ga
temperature_2m_birmingham_al
temperature_2m_huntsville_al
temperature_2m_meridian_ms
temperature_2m_mobile_al
temperature_2m_savannah_ga
relative_humidity_2m_albany_ga
relative_humidity_2m_atlanta_ga
relative_humidity_2m_birmingham_al
relative_humidity_2m_huntsville_al
relative_humidity_2m_meridian_ms
relative_humidity_2m_mobile_al
relative_humidity_2m_savannah_ga
dew_point_2m_albany_ga
dew_point_2m_atlanta_ga
dew_point_2m_birmingham_al
dew_point_2m_huntsville_al
dew_point_2m_meridian_ms
dew_point_2m_mobile_al
dew_point_2m_savannah_ga
precipitation_albany_ga
precipitation_atlanta_ga
precipitation_birmingham_al
precipitation_huntsville_al
precipitation_meridian_ms
precipitation_mobile_al
precipitation_savannah_ga
surface_pressure_albany_ga
surface_pressure_atlanta_ga
surface_pressure_birmingham_al
surface_pressure_huntsville_al
surface_pressure_meridian_ms
surface_pressure_mobile_al
su

## Data Dictionary: merged_df (SOCO Energy + Weather + Time Features)

**Target Variable:** `demand_imputed_pudl_mwh` (Electricity demand with PUDL imputation)

| Column | Category | Description |
|--------|----------|-------------|
| datetime_utc | Timestamp | Timestamp in Coordinated Universal Time (UTC). |
| datetime_local | Timestamp | Timestamp converted to local time (Eastern Time). |

### Energy System Variables (MWh)

| Column | Category | Description |
|--------|----------|-------------|
| demand_imputed_pudl_mwh | Target Variable | TARGET: Electricity demand with missing/outliers imputed by PUDL. |
| demand_imputed_pudl_mwh_lag_1h | Lagged Demand Feature | Demand from 1 hour before the current timestamp. |
| demand_imputed_pudl_mwh_lag_2h | Lagged Demand Feature | Demand from 2 hours before the current timestamp. |
| demand_imputed_pudl_mwh_lag_3h | Lagged Demand Feature | Demand from 3 hours before the current timestamp. |
| demand_imputed_pudl_mwh_lag_24h | Lagged Demand Feature | Demand from the same hour on the previous day. |
| demand_imputed_pudl_mwh_lag_48h | Lagged Demand Feature | Demand from the same hour two days earlier. |
| demand_imputed_pudl_mwh_lag_72h | Lagged Demand Feature | Demand from the same hour three days earlier. |
| demand_imputed_pudl_mwh_lag_168h | Lagged Demand Feature | Demand from the same hour one week earlier. |
| demand_imputed_pudl_mwh_rolling_mean_3h | Rolling Demand Feature | Mean demand over the previous 3 hours. |
| demand_imputed_pudl_mwh_rolling_mean_6h | Rolling Demand Feature | Mean demand over the previous 6 hours. |
| demand_imputed_pudl_mwh_rolling_mean_24h | Rolling Demand Feature | Mean demand over the previous 24 hours. |
| demand_imputed_pudl_mwh_rolling_mean_168h | Rolling Demand Feature | Mean demand over the previous 168 hours, equivalent to one week. |
| demand_imputed_pudl_mwh_rolling_max_24h | Rolling Demand Feature | Maximum demand over the previous 24 hours. |
| demand_imputed_pudl_mwh_rolling_min_24h | Rolling Demand Feature | Minimum demand over the previous 24 hours. |
| demand_imputed_pudl_mwh_rolling_std_24h | Rolling Demand Feature | Standard deviation of demand over the previous 24 hours. |

### Weather Features: City-Level

| Column | Category | Description |
|--------|----------|-------------|
| temperature_2m_albany_ga | City-Level Weather Feature | Air temperature at 2m in Albany, GA. |
| temperature_2m_atlanta_ga | City-Level Weather Feature | Air temperature at 2m in Atlanta, GA. |
| temperature_2m_birmingham_al | City-Level Weather Feature | Air temperature at 2m in Birmingham, AL. |
| temperature_2m_huntsville_al | City-Level Weather Feature | Air temperature at 2m in Huntsville, AL. |
| temperature_2m_meridian_ms | City-Level Weather Feature | Air temperature at 2m in Meridian, MS. |
| temperature_2m_mobile_al | City-Level Weather Feature | Air temperature at 2m in Mobile, AL. |
| temperature_2m_savannah_ga | City-Level Weather Feature | Air temperature at 2m in Savannah, GA. |
| relative_humidity_2m_albany_ga | City-Level Weather Feature | Relative humidity at 2m in Albany, GA. |
| relative_humidity_2m_atlanta_ga | City-Level Weather Feature | Relative humidity at 2m in Atlanta, GA. |
| relative_humidity_2m_birmingham_al | City-Level Weather Feature | Relative humidity at 2m in Birmingham, AL. |
| relative_humidity_2m_huntsville_al | City-Level Weather Feature | Relative humidity at 2m in Huntsville, AL. |
| relative_humidity_2m_meridian_ms | City-Level Weather Feature | Relative humidity at 2m in Meridian, MS. |
| relative_humidity_2m_mobile_al | City-Level Weather Feature | Relative humidity at 2m in Mobile, AL. |
| relative_humidity_2m_savannah_ga | City-Level Weather Feature | Relative humidity at 2m in Savannah, GA. |
| dew_point_2m_albany_ga | City-Level Weather Feature | Dew point temperature at 2m in Albany, GA. |
| dew_point_2m_atlanta_ga | City-Level Weather Feature | Dew point temperature at 2m in Atlanta, GA. |
| dew_point_2m_birmingham_al | City-Level Weather Feature | Dew point temperature at 2m in Birmingham, AL. |
| dew_point_2m_huntsville_al | City-Level Weather Feature | Dew point temperature at 2m in Huntsville, AL. |
| dew_point_2m_meridian_ms | City-Level Weather Feature | Dew point temperature at 2m in Meridian, MS. |
| dew_point_2m_mobile_al | City-Level Weather Feature | Dew point temperature at 2m in Mobile, AL. |
| dew_point_2m_savannah_ga | City-Level Weather Feature | Dew point temperature at 2m in Savannah, GA. |
| precipitation_albany_ga | City-Level Weather Feature | Hourly precipitation in Albany, GA. |
| precipitation_atlanta_ga | City-Level Weather Feature | Hourly precipitation in Atlanta, GA. |
| precipitation_birmingham_al | City-Level Weather Feature | Hourly precipitation in Birmingham, AL. |
| precipitation_huntsville_al | City-Level Weather Feature | Hourly precipitation in Huntsville, AL. |
| precipitation_meridian_ms | City-Level Weather Feature | Hourly precipitation in Meridian, MS. |
| precipitation_mobile_al | City-Level Weather Feature | Hourly precipitation in Mobile, AL. |
| precipitation_savannah_ga | City-Level Weather Feature | Hourly precipitation in Savannah, GA. |
| surface_pressure_albany_ga | City-Level Weather Feature | Surface pressure in Albany, GA. |
| surface_pressure_atlanta_ga | City-Level Weather Feature | Surface pressure in Atlanta, GA. |
| surface_pressure_birmingham_al | City-Level Weather Feature | Surface pressure in Birmingham, AL. |
| surface_pressure_huntsville_al | City-Level Weather Feature | Surface pressure in Huntsville, AL. |
| surface_pressure_meridian_ms | City-Level Weather Feature | Surface pressure in Meridian, MS. |
| surface_pressure_mobile_al | City-Level Weather Feature | Surface pressure in Mobile, AL. |
| surface_pressure_savannah_ga | City-Level Weather Feature | Surface pressure in Savannah, GA. |
| wind_speed_10m_albany_ga | City-Level Weather Feature | Wind speed at 10m in Albany, GA. |
| wind_speed_10m_atlanta_ga | City-Level Weather Feature | Wind speed at 10m in Atlanta, GA. |
| wind_speed_10m_birmingham_al | City-Level Weather Feature | Wind speed at 10m in Birmingham, AL. |
| wind_speed_10m_huntsville_al | City-Level Weather Feature | Wind speed at 10m in Huntsville, AL. |
| wind_speed_10m_meridian_ms | City-Level Weather Feature | Wind speed at 10m in Meridian, MS. |
| wind_speed_10m_mobile_al | City-Level Weather Feature | Wind speed at 10m in Mobile, AL. |
| wind_speed_10m_savannah_ga | City-Level Weather Feature | Wind speed at 10m in Savannah, GA. |
| shortwave_radiation_albany_ga | City-Level Weather Feature | Solar radiation in Albany, GA. |
| shortwave_radiation_atlanta_ga | City-Level Weather Feature | Solar radiation in Atlanta, GA. |
| shortwave_radiation_birmingham_al | City-Level Weather Feature | Solar radiation in Birmingham, AL. |
| shortwave_radiation_huntsville_al | City-Level Weather Feature | Solar radiation in Huntsville, AL. |
| shortwave_radiation_meridian_ms | City-Level Weather Feature | Solar radiation in Meridian, MS. |
| shortwave_radiation_mobile_al | City-Level Weather Feature | Solar radiation in Mobile, AL. |
| shortwave_radiation_savannah_ga | City-Level Weather Feature | Solar radiation in Savannah, GA. |

### Weather Features: Regional Mean

| Column | Category | Description |
|--------|----------|-------------|
| temperature_2m_regional_mean | Regional Weather Feature | Mean 2-meter air temperature across the selected SOCO-region cities. |
| relative_humidity_2m_regional_mean | Regional Weather Feature | Mean 2-meter relative humidity across the selected SOCO-region cities. |
| dew_point_2m_regional_mean | Regional Weather Feature | Mean 2-meter dew point temperature across the selected SOCO-region cities. |
| precipitation_regional_mean | Regional Weather Feature | Mean precipitation across the selected SOCO-region cities. |
| surface_pressure_regional_mean | Regional Weather Feature | Mean surface pressure across the selected SOCO-region cities. |
| wind_speed_10m_regional_mean | Regional Weather Feature | Mean 10-meter wind speed across the selected SOCO-region cities. |
| shortwave_radiation_regional_mean | Regional Weather Feature | Mean shortwave solar radiation across the selected SOCO-region cities. |

### Time-Based Features

| Column | Category | Description |
|--------|----------|-------------|
| hour | Time Feature | Hour of day (0–23). |
| day_of_week | Time Feature | Day of week (0=Monday, 6=Sunday). |
| day_of_month | Time Feature | Day of month (1–31). |
| day_of_year | Time Feature | Day of year (1–366). |
| month | Time Feature | Month (1–12). |
| quarter | Time Feature | Quarter of year (1–4). |
| year | Time Feature | Calendar year. |
| week_of_year | Time Feature | ISO week number. |

### Calendar Flags

| Column | Category | Description |
|--------|----------|-------------|
| is_weekend | Calendar Flag | Indicator for Saturday/Sunday (1=yes, 0=no). |
| is_month_start | Calendar Flag | Indicator for first day of month. |
| is_month_end | Calendar Flag | Indicator for last day of month. |
| is_quarter_start | Calendar Flag | Indicator for first day of quarter. |
| is_quarter_end | Calendar Flag | Indicator for last day of quarter. |
| is_holiday | Calendar Flag | Indicator for observed US federal holiday (1=yes, 0=no). |

### Cyclical Encodings

| Column | Category | Description |
|--------|----------|-------------|
| hour_sin | Cyclical Encoding | Sine transformation of hour (captures daily cycle). |
| hour_cos | Cyclical Encoding | Cosine transformation of hour. |
| day_of_week_sin | Cyclical Encoding | Sine transformation of day of week. |
| day_of_week_cos | Cyclical Encoding | Cosine transformation of day of week. |
| month_sin | Cyclical Encoding | Sine transformation of month (seasonality). |
| month_cos | Cyclical Encoding | Cosine transformation of month. |
| day_of_year_sin | Cyclical Encoding | Sine transformation of day of year (annual cycle). |
| day_of_year_cos | Cyclical Encoding | Cosine transformation of day of year. |

In [54]:
# export to csv
os.makedirs(os.path.dirname(merged_data_path), exist_ok=True)
merged_df.to_csv(merged_data_path, index=False)

In [56]:
# Export data dictionary to CSV

# Input markdown file
markdown_path = Path("data/soco_modeling_data_dictionary.md")

# Output CSV file
output_csv_path = Path("data/soco_modeling_data_dictionary.csv")

# Read markdown text
markdown_text = markdown_path.read_text(encoding="utf-8")

rows = []

for line in markdown_text.splitlines():
    line = line.strip()

    # Skip non-table lines
    if not line.startswith("|") or not line.endswith("|"):
        continue

    # Skip header rows
    if "Column" in line and "Description" in line:
        continue

    # Skip markdown separator rows, including 2-column or 3-column tables
    if re.match(r"^\|\s*:?-+:?\s*(\|\s*:?-+:?\s*)+\|$", line):
        continue

    # Split row into cells
    parts = [part.strip() for part in line.strip("|").split("|")]

    # Support 3-column format: Column | Category | Description
    if len(parts) == 3:
        column, category, description = parts

    # Support older 2-column format: Column | Description
    elif len(parts) == 2:
        column, description = parts
        category = "Uncategorized"

    else:
        continue

    # Remove markdown bold/backticks
    column = re.sub(r"[*`]", "", column).strip()
    category = re.sub(r"[*`]", "", category).strip()
    description = re.sub(r"[*`]", "", description).strip()

    # Skip empty rows
    if not column or not description:
        continue

    rows.append({
        "column": column,
        "category": category,
        "description": description
    })

# Create dataframe
data_dictionary_df = pd.DataFrame(rows)

# Optional: remove duplicate rows, if any
data_dictionary_df = (
    data_dictionary_df
    .drop_duplicates(subset=["column"])
    .reset_index(drop=True)
)

# Export CSV
data_dictionary_df.to_csv(output_csv_path, index=False)

print(f"Exported {len(data_dictionary_df)} rows to {output_csv_path}")
data_dictionary_df

Exported 95 rows to data\soco_modeling_data_dictionary.csv


,column,category,description
0,datetime_utc,Timestamp,Timestamp in Coordinated Universal Time (UTC).
1,datetime_local,Timestamp,Timestamp converted to local time (Eastern Time).
2,demand_imputed_pudl_mwh,Target Variable,TARGET: Electricity demand with missing/outlie...
3,demand_imputed_pudl_mwh_lag_1h,Lagged Demand Feature,Demand from 1 hour before the current timestamp.
4,demand_imputed_pudl_mwh_lag_2h,Lagged Demand Feature,Demand from 2 hours before the current timestamp.
...,...,...,...
90,day_of_week_cos,Cyclical Encoding,Cosine transformation of day of week.
91,month_sin,Cyclical Encoding,Sine transformation of month (seasonality).
92,month_cos,Cyclical Encoding,Cosine transformation of month.
93,day_of_year_sin,Cyclical Encoding,Sine transformation of day of year (annual cyc...
